# HeartLang on PTB-XL: reproducing the linear probe with PyHealth

<a href="https://colab.research.google.com/github/sunlabuiuc/PyHealth/blob/master/examples/heartlang_ptbxl_linear_probe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Paper:** Jiarui Jin, Haoyu Wang, Hongyan Li, Jun Li, Jiahui Pan and Shenda Hong.
*Reading Your Heart: Learning ECG Words and Sentences via Pre-training ECG Language Model.*
ICLR 2025. [OpenReview](https://openreview.net/forum?id=6Hz1Ko087B) ·
[code](https://github.com/PKUDigitalHealth/HeartLang) ·
[checkpoints](https://huggingface.co/PKUDigitalHealth/HeartLang)

HeartLang reads an ECG as a *sentence*: each heartbeat of each lead, cut around the QRS
complexes of lead I, is a *word*. A transformer pre-trained on MIMIC-IV-ECG with masked
heartbeat modelling encodes the sentence. The paper evaluates the frozen encoder with a
**linear probe** on PTB-XL and reports macro AUROC (Table 2, 100% of the training labels).

This notebook reproduces that experiment with PyHealth:

1. download PTB-XL (open access) and the authors' pre-trained checkpoint,
2. build the HeartLang inputs with `PTBXLHeartLangClassification` (100 Hz, the PTB-XL
   benchmark's labels and standardization),
3. split by the benchmark folds (1-8 train, 9 validation, 10 test),
4. train a linear head on the frozen encoder and report test macro AUROC next to the paper.

Everything runs on fully open data. The four PTB-XL subsets took about 50 minutes on a
laptop (Apple M-series, MPS). Most of that is building and caching the inputs of each
subset (QRS detection on every ECG, which runs on the CPU); the probes take minutes. We
have not timed a full Colab run: with a T4 GPU but only two CPU cores, expect roughly one
to two hours. `FRACTION = 0.1` below runs about a tenth of the data as a quick check.

**Our run (2026-11-03, full PTB-XL, Apple MPS), test macro AUROC (%):**

| Subset | Classes | PyHealth | Paper (Table 2) |
|---|---|---|---|
| superdiagnostic | 5 | 88.54 | 87.52 |
| subdiagnostic | 23 | 89.29 | 88.91 |
| form | 19 | 80.76 | 80.23 |
| rhythm | 12 | 89.57 | 90.34 |

In [ ]:
# PyHealth with HeartLang, from GitHub master for now. Once HeartLang is in a
# PyPI release, use `pip install "pyhealth[ptbxl]"` instead.
!pip install -q "pyhealth[ptbxl] @ git+https://github.com/sunlabuiuc/PyHealth"

## Settings

`LABEL_SETS` chooses the PTB-XL subsets to evaluate. `FRACTION` below 1 keeps that share of
every fold for a quick smoke test; the reported numbers need `FRACTION = 1.0`.

In [ ]:
import os

import torch

LABEL_SETS = ["superdiagnostic", "subdiagnostic", "form", "rhythm"]
FRACTION = 1.0          # share of each PTB-XL fold to use (1.0 = full dataset)
DATA_ROOT = "ptb-xl"    # where PTB-XL 1.0.3 is downloaded
CACHE_DIR = "pyhealth_cache"
DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")

# Linear-probe hyperparameters from the authors' PTB-XL script
# (scripts/finetune/PTBXL/12Leads/*/base_*_linear_prob.sh)
EPOCHS, WARMUP_EPOCHS, BATCH_SIZE, LR, WEIGHT_DECAY = 100, 10, 256, 5e-3, 0.05

PAPER_MACRO_AUROC = {  # Table 2, linear probe, 100% of the training labels
    "superdiagnostic": 87.52, "subdiagnostic": 88.91, "form": 80.23, "rhythm": 90.34,
}
print(DEVICE)

## Download PTB-XL 1.0.3

The 100 Hz records come from PhysioNet's open-data mirror on AWS (about 550 MB). With
`FRACTION < 1` only a stratified share of every fold is downloaded.

In [ ]:
import concurrent.futures
import time
import urllib.request

import pandas as pd

MIRROR = "https://physionet-open.s3.amazonaws.com/ptb-xl/1.0.3/"


def fetch(relative_path, local_name=None, attempts=5):
    """Download one file, retrying dropped connections.

    The file is written to a temporary name and renamed when complete, so an
    interrupted download is never mistaken for a finished one on a rerun.
    """
    target = os.path.join(DATA_ROOT, local_name or relative_path)
    if os.path.exists(target):
        return
    os.makedirs(os.path.dirname(target) or ".", exist_ok=True)
    for attempt in range(attempts):
        try:
            urllib.request.urlretrieve(MIRROR + relative_path, target + ".part")
            os.replace(target + ".part", target)
            return
        except OSError:
            if attempt == attempts - 1:
                raise
            time.sleep(2 ** attempt)


os.makedirs(DATA_ROOT, exist_ok=True)
fetch("scp_statements.csv")
fetch("ptbxl_database.csv", local_name="ptbxl_database_full.csv")

database = pd.read_csv(os.path.join(DATA_ROOT, "ptbxl_database_full.csv"), index_col=0)
if FRACTION < 1:
    database = database.groupby("strat_fold", group_keys=False).sample(
        frac=FRACTION, random_state=0)
# PTBXLDataset reads ptbxl_database.csv: write the selected records there
database.to_csv(os.path.join(DATA_ROOT, "ptbxl_database.csv"))

files = [f + ext for f in database.filename_lr for ext in (".hea", ".dat")]
with concurrent.futures.ThreadPoolExecutor(8) as pool:
    list(pool.map(fetch, files))
print(f"{len(database)} ECGs downloaded")

## Build the HeartLang inputs

`PTBXLHeartLangClassification` standardizes every ECG with one mean and standard deviation
(those of all PTB-XL training samples, as in the PTB-XL benchmark the paper follows), detects
the QRS complexes on lead I with `wfdb`'s XQRS detector, and cuts the 12 leads into 96-sample
heartbeat words: up to 256 words per ECG, each with a lead and a one-second time index.
Labels use the PTB-XL benchmark's aggregation (`scp_codes` with any likelihood); ECGs without
a label in the subset are dropped.

In [ ]:
from pyhealth.datasets import PTBXLDataset, split_by_strat_fold
from pyhealth.tasks import PTBXLHeartLangClassification

dataset = PTBXLDataset(root=DATA_ROOT, sampling_rate=100, cache_dir=CACHE_DIR)


def build(label_set):
    task = PTBXLHeartLangClassification(
        scp_statements_path=dataset.scp_statements_path, label_set=label_set)
    samples = dataset.set_task(task)
    return samples, split_by_strat_fold(samples)


samples, (train_ds, val_ds, test_ds) = build(LABEL_SETS[0])
example = samples[0]
print({k: tuple(v.shape) if hasattr(v, "shape") else v for k, v in example.items()})
print(len(train_ds), len(val_ds), len(test_ds))

## Load the pre-trained model

`HeartLang(..., pretrained=True)` downloads the authors' MIMIC-IV-ECG checkpoint
(`checkpoints/pretrain/MIMIC-IV/checkpoint-200.pth`, about 535 MB) and checks that every
weight matches. `freeze_backbone=True` leaves only the linear head trainable.

In [ ]:
from pyhealth.models import HeartLang

model = HeartLang(samples, pretrained=True, freeze_backbone=True).to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable parameters: {trainable}")

## Linear probe

The encoder is frozen, so its [CLS] features are computed once and the linear head is trained
on them: AdamW, learning rate 5e-3 with 10 warm-up epochs and cosine decay, weight decay 0.05,
batch size 256, 100 epochs, binary cross-entropy. The epoch with the best validation macro
AUROC is evaluated on the test fold.

One difference from the authors' script: there the frozen encoder stays in training mode, so
its dropout is active while the head is trained. Here features come from the encoder in
evaluation mode, the usual linear-probe setup, which makes the probe a few seconds per epoch.

In [ ]:
import math

import numpy as np
from sklearn.metrics import roc_auc_score

from pyhealth.datasets import get_dataloader


@torch.no_grad()
def features(model, ds):
    model.eval()
    xs, ys = [], []
    for batch in get_dataloader(ds, batch_size=BATCH_SIZE, shuffle=False):
        xs.append(model.embed(**batch).cpu())
        ys.append(batch["labels"].float())
    return torch.cat(xs), torch.cat(ys)


def macro_auroc(y, logits):
    y, p = y.numpy(), torch.sigmoid(logits).numpy()
    keep = (y.sum(0) > 0) & (y.sum(0) < len(y))  # classes with both outcomes
    return 100 * roc_auc_score(y[:, keep], p[:, keep], average="macro")


def linear_probe(model, train_ds, val_ds, test_ds, seed=0):
    torch.manual_seed(seed)
    (xtr, ytr), (xva, yva), (xte, yte) = (
        features(model, d) for d in (train_ds, val_ds, test_ds))
    head = torch.nn.Linear(xtr.shape[1], ytr.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    steps_per_epoch = math.ceil(len(xtr) / BATCH_SIZE)
    total, warmup = EPOCHS * steps_per_epoch, WARMUP_EPOCHS * steps_per_epoch
    schedule = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warmup
        if s < warmup else 0.5 * (1 + math.cos(math.pi * (s - warmup) / (total - warmup))))
    best = (-1.0, None)
    for epoch in range(EPOCHS):
        head.train()
        for idx in torch.randperm(len(xtr)).split(BATCH_SIZE):
            loss = torch.nn.functional.binary_cross_entropy_with_logits(
                head(xtr[idx].to(DEVICE)), ytr[idx].to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step(); schedule.step()
        head.eval()
        with torch.no_grad():
            val = macro_auroc(yva, head(xva.to(DEVICE)).cpu())
        if val > best[0]:
            best = (val, {k: v.clone() for k, v in head.state_dict().items()})
    head.load_state_dict(best[1])
    model.mlp_head.load_state_dict(best[1])  # the PyHealth model now holds the probe
    with torch.no_grad():
        test = macro_auroc(yte, head(xte.to(DEVICE)).cpu())
    return best[0], test


results = {}
for label_set in LABEL_SETS:
    if label_set != LABEL_SETS[0]:
        samples, (train_ds, val_ds, test_ds) = build(label_set)
        model = HeartLang(samples, pretrained=True, freeze_backbone=True).to(DEVICE)
    val, test = linear_probe(model, train_ds, val_ds, test_ds)
    results[label_set] = test
    print(f"{label_set:16s} classes={samples.output_processors['labels'].size():3d} "
          f"val={val:.2f} test={test:.2f}")

## Results

Test macro AUROC (%) of the linear probe, next to the paper's Table 2 (100% of the labels).

In [ ]:
table = pd.DataFrame({
    "PyHealth (this notebook)": pd.Series(results).round(2),
    "Paper (Table 2)": pd.Series(PAPER_MACRO_AUROC),
})
table["difference"] = (table.iloc[:, 0] - table.iloc[:, 1]).round(2)
table.loc[[k for k in LABEL_SETS if k in results]]

## Using HeartLang in your own PyHealth pipeline

The same model trains end to end with PyHealth's `Trainer`, for example to fine-tune all
layers (`freeze_backbone=False`) instead of probing:

```python
from pyhealth.trainer import Trainer

model = HeartLang(samples, pretrained=True, freeze_backbone=False)
trainer = Trainer(model=model, metrics=["roc_auc_macro"], device=DEVICE)
trainer.train(
    train_dataloader=get_dataloader(train_ds, batch_size=64, shuffle=True),
    val_dataloader=get_dataloader(val_ds, batch_size=64),
    epochs=10, optimizer_params={"lr": 1e-4}, monitor="roc_auc_macro",
)
print(trainer.evaluate(get_dataloader(test_ds, batch_size=64)))
```

`ecg_to_heartbeat_sentence` turns a 12-lead ECG array into the model's inputs, so other
ECG datasets only need a task that calls it. Resample to 100 Hz and crop to at most 15 s
first: the published model was trained on 10 s ECGs at 100 Hz, and its time embedding
covers 15 s (longer recordings raise an error).

## What to check (for the HeartLang authors)

- **Tokenization:** `pyhealth.models.heartlang.ecg_to_heartbeat_sentence` ports
  `QRSTokenizer` (XQRS on lead I, words of 96 samples centred on each QRS, up to 256 words,
  lead index + 1 and second + 1 as the spatial and temporal indices).
- **Standardization:** one global mean and standard deviation over the PTB-XL training folds
  (`PTBXL_100HZ_MEAN`, `PTBXL_100HZ_STD`), as in `ecg_ptbxl_benchmarking`.
- **Labels and split:** PyHealth's PTB-XL label aggregation (no likelihood threshold) and
  folds 1-8 / 9 / 10. The number of classes per subset is printed above
  (paper: super 5, sub 23, form 19, rhythm 12).
- **Encoder:** `HeartLangBackbone` keeps the parameter names of `ST_ECGFormer`, so the
  published checkpoint loads with no missing or unexpected keys.
- **Probe:** features from the frozen encoder in evaluation mode (no dropout); the original
  script keeps dropout on while the head trains.